# Day 9 — LSTM Demand Forecasting | RetailMind AI

**Objective:** Forecast next-day store sales using the Rossmann Store Sales dataset.

This notebook:
1. Loads and cleans daily store sales.
2. Creates 28-day look-back windows.
3. Scales each store using its training-period maximum.
4. Holds out the most recent 42 days per store.
5. Trains a shared LSTM model.
6. Evaluates MAE/RMSE in original sales units.
7. Plots actual vs predicted sales and saves the model + scaling metadata.

> **Dataset:** Download `train.csv` from the [Rossmann Store Sales Kaggle competition](https://www.kaggle.com/competitions/rossmann-store-sales/data) and place it at `RetailMind/data/rossmann/train.csv`.

Run this notebook from `RetailMind/notebooks/` or adjust `DATA_PATH` below if your layout differs.

## 1. Imports and configuration

Required libraries: TensorFlow/Keras, pandas, and Matplotlib for plotting. NumPy is used only for array handling.

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("Pandas:", pd.__version__)

# Notebook is expected to be in RetailMind/notebooks/
ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

DATA_PATH = ROOT_DIR / "data" / "rossmann" / "train.csv"
MODEL_DIR = ROOT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "demand_lstm.keras"
METADATA_PATH = MODEL_DIR / "demand_scalers.pkl"
PREDICTIONS_PATH = ROOT_DIR / "data" / "demand_predictions.csv"

LOOKBACK = 28
TEST_DAYS = 42
EPOCHS = 30
BATCH_SIZE = 256
RANDOM_SEED = 42

tf.keras.utils.set_random_seed(RANDOM_SEED)

print("Project root:", ROOT_DIR)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

I0000 00:00:1794766494.667784   13259 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794766495.058792   13259 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794766498.491852   13259 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow: 2.22.0-rc0
Pandas: 3.1.0
Project root: /home/pankaj/Documents/LearnLytica Assignments/LearnLytica Incremental Project
Dataset path: /home/pankaj/Documents/LearnLytica Assignments/LearnLytica Incremental Project/data/rossmann/train.csv
Dataset exists: True


## 2. Load and inspect the dataset

We begin with `Store`, `Date`, and `Sales`. This baseline deliberately uses sales history only; promotions, holidays, and customer counts can be added in a later increment.

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Download Rossmann train.csv "
        "and place it in data/rossmann/train.csv."
    )

df = pd.read_csv(DATA_PATH)

required_columns = {"Store", "Date", "Sales"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")

df = df.dropna(subset=["Store", "Date", "Sales"]).copy()
df["Store"] = df["Store"].astype(int)

# Aggregate in case a store/date appears more than once.
daily_sales = (
    df.groupby(["Store", "Date"], as_index=False)["Sales"]
      .sum()
      .sort_values(["Store", "Date"])
)

display(daily_sales.head())
print("Rows:", len(daily_sales))
print("Stores:", daily_sales["Store"].nunique())
print("Date range:", daily_sales["Date"].min(), "to", daily_sales["Date"].max())

/tmp/ipykernel_13259/3054922927.py:7: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


,Store,Date,Sales
0,1,2013-01-01,0
1,1,2013-01-02,5530
2,1,2013-01-03,4327
3,1,2013-01-04,4486
4,1,2013-01-05,4997


Rows: 1017209
Stores: 1115
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00


## 3. Build one daily time series per store

Resampling ensures each store has a continuous daily index. Missing dates are filled with zero sales for this baseline.

**Note:** Rossmann's `Sales` is zero on closed days. Filling missing calendar dates with zero is a simplifying assumption; it should be revisited if missing dates represent unavailable data rather than store closures.

In [3]:
store_series = {}

for store_id, group in daily_sales.groupby("Store"):
    series = (
        group.set_index("Date")["Sales"]
             .sort_index()
             .resample("D")
             .sum()
    )
    store_series[int(store_id)] = series.astype("float32")

print("Prepared store series:", len(store_series))
example_store = next(iter(store_series))
display(store_series[example_store].head())
print("Example store:", example_store, "| observations:", len(store_series[example_store]))

Prepared store series: 1115


Date
2013-01-01       0.0
2013-01-02    5530.0
2013-01-03    4327.0
2013-01-04    4486.0
2013-01-05    4997.0
Freq: D, Name: Sales, dtype: float32

Example store: 1 | observations: 942


## 4. Create supervised windows and chronological splits

For every store:
- **Input (`X`):** sales from the previous 28 days.
- **Target (`y`):** sales on the following day.
- **Training period:** all eligible target dates before the final 42 days.
- **Test period:** the final 42 target dates.

Each store is scaled by its **training-period maximum**. The same scale is used for its train and test windows, preventing the test period from influencing the scaling factor.

In [4]:
def create_windows(store_series, lookback=LOOKBACK, test_days=TEST_DAYS):
    X_train_parts, y_train_parts = [], []
    X_test_parts, y_test_parts = [], []

    test_dates, test_stores = [], []
    scalers = {}

    for store_id, series in store_series.items():
        values = series.to_numpy(dtype=np.float32)

        split_idx = len(values) - test_days
        if split_idx <= lookback:
            print(f"Skipping store {store_id}: not enough observations")
            continue

        # Fit scale on training period only.
        scale = float(np.max(values[:split_idx]))
        if not np.isfinite(scale) or scale <= 0:
            scale = 1.0

        scalers[int(store_id)] = scale
        normalized = values / scale

        # Each row contains 28 input values followed by the next-day target.
        windows = np.lib.stride_tricks.sliding_window_view(
            normalized, window_shape=lookback + 1
        )

        X = windows[:, :-1]
        y = windows[:, -1]

        # Window index i predicts original time index i + lookback.
        train_count = split_idx - lookback

        X_train_parts.append(X[:train_count, :, np.newaxis])
        y_train_parts.append(y[:train_count])

        X_test_parts.append(X[train_count:, :, np.newaxis])
        y_test_parts.append(y[train_count:])

        test_dates.extend(series.index[split_idx:])
        test_stores.extend([int(store_id)] * (len(values) - split_idx))

    if not X_train_parts or not X_test_parts:
        raise ValueError("No valid train/test windows were created.")

    X_train = np.concatenate(X_train_parts, axis=0).astype("float32")
    y_train = np.concatenate(y_train_parts, axis=0).astype("float32")
    X_test = np.concatenate(X_test_parts, axis=0).astype("float32")
    y_test = np.concatenate(y_test_parts, axis=0).astype("float32")

    return (
        X_train, y_train, X_test, y_test,
        scalers, pd.to_datetime(test_dates), np.asarray(test_stores)
    )


(
    X_train, y_train, X_test, y_test,
    scalers, test_dates, test_stores
) = create_windows(store_series)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("Number of store scalers:", len(scalers))

X_train: (972279, 28, 1)
y_train: (972279,)
X_test : (46830, 28, 1)
y_test : (46830,)
Number of store scalers: 1115


## 5. Build the LSTM model

The model receives a tensor shaped `(batch, 28, 1)` and outputs one normalized next-day sales value.

In [5]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(LOOKBACK, 1)),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()

E0000 00:00:1794766524.016486   13259 cuda_executor.cc:1690] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:68

W0000 00:00:1794766524.016974   13367 cuda_executor.cc:1710] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:77
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:107
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:112

W0000 00:00:1794766524.041865   13259 dlopen_checker.cc:38] Could not load dynamic library 'libcudart.so.12'; dlerror: libcudart.so.12: cannot open shared object file: No such file or directory
W0000 00:00:1794766524.041897   1325

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 28, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 28, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,857 (116.63 KB)

 Trainable params: 29,857 (116.63 KB)

 Non-trainable params: 0 (0.00 B)

## 6. Train the model

The test set remains untouched during training. We reserve the final 10% of the training windows for validation. Since windows are grouped store by store, this validation split is a training-monitoring split rather than a separate global date holdout.

In [6]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stopping],
    shuffle=True,
    verbose=1
)

Epoch 1/30


W0000 00:00:1794766527.339851   13259 cpu_allocator_impl.cc:174] Allocation of 98005712 exceeds 10% of free system memory.


 119/3419 ━━━━━━━━━━━━━━━━━━━━ 4:03 74ms/step - loss: 0.0481 - mae: 0.1673

KeyboardInterrupt: 

In [ ]:
# Plot training and validation loss
history_df = pd.DataFrame(history.history)

ax = history_df[["loss", "val_loss"]].plot(
    figsize=(10, 5),
    title="LSTM Training and Validation Loss"
)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
plt.tight_layout()
plt.show()

## 7. Evaluate on the most recent 42 days

Predictions and actual values are converted back to original sales units using the corresponding store's training-period scale.

- **MAE:** average absolute error in sales units.
- **RMSE:** square root of mean squared error; it penalizes larger errors more strongly.

In [ ]:
predictions_normalized = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).reshape(-1)

actual_sales = np.empty(len(y_test), dtype=np.float32)
predicted_sales = np.empty(len(predictions_normalized), dtype=np.float32)

for i, store_id in enumerate(test_stores):
    scale = scalers[int(store_id)]
    actual_sales[i] = y_test[i] * scale
    predicted_sales[i] = predictions_normalized[i] * scale

mae = float(np.mean(np.abs(actual_sales - predicted_sales)))
rmse = float(np.sqrt(np.mean((actual_sales - predicted_sales) ** 2)))

print(f"Test MAE : {mae:.2f} sales units")
print(f"Test RMSE: {rmse:.2f} sales units")

results = pd.DataFrame({
    "Date": test_dates,
    "Store": test_stores,
    "Actual": actual_sales,
    "Predicted": predicted_sales
}).sort_values(["Store", "Date"])

display(results.head(10))

## 8. Plot actual vs predicted demand

The plot below shows the first store represented in the test results. This is a sanity check, not a complete assessment of every store.

In [ ]:
store_to_plot = int(results["Store"].iloc[0])

store_results = (
    results[results["Store"] == store_to_plot]
    .sort_values("Date")
    .set_index("Date")
)

ax = store_results[["Actual", "Predicted"]].plot(
    figsize=(14, 6),
    title=f"Actual vs Predicted Daily Sales — Store {store_to_plot}"
)
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
plt.tight_layout()
plt.show()

## 9. Save the model, scaling metadata, and predictions

The model alone is not sufficient for inference: we also need the per-store scaling values and look-back length to prepare future inputs consistently.

In [ ]:
model.save(MODEL_PATH)

metadata = {
    "scalers": scalers,
    "lookback": LOOKBACK,
    "model_type": "shared_store_lstm",
    "target": "Sales",
    "frequency": "daily"
}

with open(METADATA_PATH, "wb") as file:
    pickle.dump(metadata, file)

PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(PREDICTIONS_PATH, index=False)

print("Saved model      :", MODEL_PATH)
print("Saved metadata   :", METADATA_PATH)
print("Saved predictions:", PREDICTIONS_PATH)

## 10. Reload and verify the saved model

This confirms that the saved Keras model can be loaded and used for inference.

In [ ]:
loaded_model = tf.keras.models.load_model(MODEL_PATH)

sample_prediction = loaded_model.predict(X_test[:1], verbose=0)[0, 0]
sample_store = int(test_stores[0])
sample_sales_prediction = float(sample_prediction * scalers[sample_store])

print("Reloaded model successfully.")
print(f"Example next-day sales prediction for store {sample_store}: "
      f"{sample_sales_prediction:.2f}")

## 11. Day 9 summary and limitations

### Deliverables
- Shared LSTM trained across store-level daily sales series.
- 28-day input window and one-day-ahead target.
- Final 42 days held out per store for testing.
- MAE and RMSE reported in original sales units.
- Actual-vs-predicted plot.
- Saved model, per-store scaling metadata, and test predictions.

### Important limitations
1. This is a sales-only baseline; promotions, holidays, and other calendar signals are not included.
2. Zero sales can mean a store was closed, so the model learns closure patterns as part of the sales series.
3. The evaluation is rolling one-day-ahead: each test prediction uses the previous 28 **actual** sales values. It is not a 42-day recursive forecast.
4. The shared model learns across stores, but uses a separate scale for each store.
5. Sales are not necessarily identical to unconstrained customer demand; stock availability and store closures can affect observed sales.

### Next improvement
Add known future features such as promotions, school holidays, and state holidays, then compare the feature-aware model against this baseline.